<a href="https://colab.research.google.com/github/edjnolasco/transport-ml-rd/blob/feature%2Fgreen-ai/notebooks/practica3_transporte_greenai_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctica 3 – Mejora del algoritmo de clasificación (Green AI)

**Autor:** Edwin José Nolasco  
**Asignatura:** INF-9005-C2 Algoritmos de Clasificación en Machine Learning  
**Programa:** Doctorado en Inteligencia Artificial y Aprendizaje Automático (UASD–UMH)  
**Catedrática:** María Belén Pérez Sánchez, PhD

---

## Contexto

Este notebook corresponde a la **Práctica 3**, en la cual se extiende el trabajo desarrollado en la Práctica 2.

El objetivo es analizar el modelo de clasificación previamente implementado incorporando una nueva dimensión de evaluación: la eficiencia computacional, en el marco del enfoque de Green AI.

En particular, se estudia el impacto de:

- el tamaño de la muestra  
- el número de variables  
- la configuración del modelo  

sobre el rendimiento predictivo y el coste de entrenamiento.

---

## Propósito del notebook

Este notebook tiene como objetivo:

- ejecutar los experimentos  
- generar métricas y resultados  
- producir evidencia gráfica reproducible  

El análisis e interpretación de los resultados se presenta en el documento de informe correspondiente.


In [1]:
# ============================================================
# Preparación del repositorio en Colab
# Clona el repositorio solo si aún no existe.
# ============================================================

import os

REPO_URL = "https://github.com/edjnolasco/transport-ml-rd.git"
REPO_PATH = "/content/transport-ml-rd"

if not os.path.exists(REPO_PATH):
    !git clone {REPO_URL} {REPO_PATH}
else:
    print("Repositorio ya disponible en:", REPO_PATH)


Cloning into '/content/transport-ml-rd'...
remote: Enumerating objects: 449, done.
remote: Counting objects: 100% (186/186), done.
remote: Compressing objects: 100% (148/148), done.
remote: Total 449 (delta 103), reused 29 (delta 26), pack-reused 263 (from 2)
Receiving objects: 100% (449/449), 985.46 KiB | 8.08 MiB/s, done.
Resolving deltas: 100% (199/199), done.


In [2]:
# ============================================================
# Configuración general del entorno
# ============================================================

import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROJECT_ROOT = Path("/content/transport-ml-rd").resolve()
sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Existe el proyecto:", PROJECT_ROOT.exists())
if PROJECT_ROOT.exists():
    print("Contenido raíz del proyecto:")
    print(os.listdir(PROJECT_ROOT))


PROJECT_ROOT: /content/transport-ml-rd
Existe el proyecto: True
Contenido raíz del proyecto:
['LICENSE', 'notebooks', '.github', '.gitignore', 'src', 'reports', 'README.md', 'coverage.svg', 'docs', 'main.py', 'requirements.txt', 'ruff.toml', 'tests', 'pyproject.toml', '.git', 'README_tests.md', 'data']


In [3]:
# ============================================================
# Importaciones del proyecto
# ============================================================

from src.config import build_config
from src.data_processing import load_dataset, basic_cleaning
from src.features import split_features_target

config = build_config(PROJECT_ROOT)

print("Archivo de entrada esperado por config:", config.input_file)
print("Archivo procesado esperado por config:", config.processed_file)


Archivo de entrada esperado por config: /content/transport-ml-rd/data/raw/transport_data.csv
Archivo procesado esperado por config: /content/transport-ml-rd/data/processed/transport_clean.csv


## Carga y limpieza de datos

Este bloque reutiliza la lógica central de la práctica 2, pero deja los experimentos Green AI en una notebook separada.


In [4]:
# ============================================================
# Carga y limpieza de datos
# ============================================================

df_raw = load_dataset(config.input_file)
print("Dataset original:", df_raw.shape)

df_clean = basic_cleaning(df_raw.copy())
print("Dataset limpio:", df_clean.shape)

display(df_clean.head())

print("\nColumnas disponibles:")
for col in df_clean.columns:
    print("-", col)

if "fallecidos" not in df_clean.columns:
    raise ValueError(
        "La columna base 'fallecidos' no existe en el dataset limpio. "
        "Revisa el nombre real de la variable objetivo base."
    )


Dataset original: (351, 3)
Dataset limpio: (351, 3)


,provincia,year,fallecidos
0,Azua,2016,64
1,Bahoruco,2016,21
2,Barahona,2016,34
3,Dajabón,2016,21
4,Distrito Nacional,2016,65



Columnas disponibles:
- provincia
- year
- fallecidos


In [5]:
# ============================================================
# Creación del target y separación X / y
# Se define una clasificación binaria por severidad relativa.
# ============================================================

BASE_COLUMN = "fallecidos"
TARGET_COLUMN = "fallecidos_clase"

median_value = df_clean[BASE_COLUMN].median()
print("Mediana de fallecidos:", median_value)

df_clean[TARGET_COLUMN] = (df_clean[BASE_COLUMN] > median_value).astype(int)

print("\nDistribución nueva de clases:")
print(df_clean[TARGET_COLUMN].value_counts(dropna=False))

X, y = split_features_target(df_clean, target_column=TARGET_COLUMN)

# Evitar fuga de información: eliminar la variable base del conjunto predictor
if BASE_COLUMN in X.columns:
    X = X.drop(columns=[BASE_COLUMN])

# Normalizar y a Series para validaciones consistentes
y = pd.Series(y).reset_index(drop=True)

print("\nForma de X:", X.shape)
print("Longitud de y:", len(y))
print("Distribución final de y:")
print(y.value_counts(dropna=False))

if y.nunique() < 2:
    raise ValueError(
        "La variable objetivo quedó con una sola clase. "
        "No es posible entrenar modelos de clasificación."
    )


Mediana de fallecidos: 40.0

Distribución nueva de clases:
fallecidos_clase
0    176
1    175
Name: count, dtype: int64

Forma de X: (351, 2)
Longitud de y: 351
Distribución final de y:
fallecidos_clase
0    176
1    175
Name: count, dtype: int64


In [6]:
# ============================================================
# Conversión segura para experimentos Green AI
# Esta transformación no altera tu pipeline base original.
# ============================================================

X_green = X.copy()
X_green = pd.get_dummies(X_green, drop_first=True)
X_green = X_green.fillna(0)
X_green.columns = [str(col) for col in X_green.columns]

print("Forma de X_green:", X_green.shape)
print("Primeras columnas:")
display(pd.Series(X_green.columns[:15]))
display(X_green.head())


Forma de X_green: (351, 63)
Primeras columnas:


,0
0,year
1,provincia_Azua
2,provincia_BAHORUCO
3,provincia_BARAHONA
4,provincia_Bahoruco
5,provincia_Barahona
6,provincia_DAJABON
7,provincia_DISTRITO NACIONAL
8,provincia_DUARTE
9,provincia_Dajabón


,year,provincia_Azua,provincia_BAHORUCO,provincia_BARAHONA,provincia_Bahoruco,provincia_Barahona,provincia_DAJABON,provincia_DISTRITO NACIONAL,provincia_DUARTE,provincia_Dajabón,...,provincia_San Cristóbal,provincia_San José de Ocoa,provincia_San Juan,provincia_San Pedro de Macorís,provincia_Santiago,provincia_Santiago Rodríguez,provincia_Santo Domingo,provincia_Sánchez Ramírez,provincia_VALVERDE,provincia_Valverde
0,2016,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,2016,False,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,2016,False,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,2016,False,False,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
4,2016,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


## Función central de evaluación

La siguiente función mide el tiempo de entrenamiento y calcula métricas predictivas para cada configuración experimental.


In [7]:
# ============================================================
# Función base de entrenamiento y evaluación
# Incluye validaciones tempranas del target.
# ============================================================

def train_and_evaluate_model(
    model,
    X_train,
    X_test,
    y_train,
    y_test,
    experiment_name,
    sample_fraction,
    n_features,
    extra_info=None,
):
    y_train = pd.Series(y_train).reset_index(drop=True)
    y_test = pd.Series(y_test).reset_index(drop=True)

    if y_train.nunique() < 2:
        raise ValueError(
            f"El conjunto de entrenamiento para '{experiment_name}' contiene una sola clase: "
            f"{sorted(y_train.unique().tolist())}"
        )

    if y_test.nunique() < 2:
        print(
            f"Advertencia: el conjunto de prueba para '{experiment_name}' contiene una sola clase: "
            f"{sorted(y_test.unique().tolist())}. Algunas métricas pueden no ser informativas."
        )

    local_model = clone(model)

    start_time = time.perf_counter()
    local_model.fit(X_train, y_train)
    end_time = time.perf_counter()

    train_time = end_time - start_time
    y_pred = local_model.predict(X_test)

    result = {
        "experiment": experiment_name,
        "model": local_model.__class__.__name__,
        "sample_fraction": sample_fraction,
        "n_features": n_features,
        "train_time_sec": train_time,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_weighted": precision_score(
            y_test, y_pred, average="weighted", zero_division=0
        ),
        "recall_weighted": recall_score(
            y_test, y_pred, average="weighted", zero_division=0
        ),
        "f1_weighted": f1_score(
            y_test, y_pred, average="weighted", zero_division=0
        ),
    }

    try:
        if hasattr(local_model, "predict_proba"):
            y_proba = local_model.predict_proba(X_test)
            if y_proba.shape[1] == 2:
                result["roc_auc"] = roc_auc_score(y_test, y_proba[:, 1])
            else:
                result["roc_auc"] = roc_auc_score(
                    y_test, y_proba, multi_class="ovr", average="weighted"
                )
        else:
            result["roc_auc"] = np.nan
    except Exception:
        result["roc_auc"] = np.nan

    if extra_info:
        result.update(extra_info)

    return result


In [8]:
# ============================================================
# Modelo base de referencia
# Se usa LogisticRegression como baseline interpretable y eficiente.
# ============================================================

base_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
)

print(base_model)


LogisticRegression(max_iter=1000, random_state=42)


In [9]:
# ============================================================
# Split base común
# ============================================================

X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_green,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Train:", X_train_base.shape)
print("Test :", X_test_base.shape)

print("\nDistribución de y_train_base:")
print(pd.Series(y_train_base).value_counts(dropna=False))

print("\nDistribución de y_test_base:")
print(pd.Series(y_test_base).value_counts(dropna=False))


Train: (280, 63)
Test : (71, 63)

Distribución de y_train_base:
fallecidos_clase
1    140
0    140
Name: count, dtype: int64

Distribución de y_test_base:
fallecidos_clase
0    36
1    35
Name: count, dtype: int64


## Experimento 0 - Línea base

Esta configuración representa el punto de partida para comparar los demás experimentos de Green AI.


In [10]:
# ============================================================
# Experimento 0 - Línea base
# ============================================================

green_results = []

baseline_result = train_and_evaluate_model(
    model=base_model,
    X_train=X_train_base,
    X_test=X_test_base,
    y_train=y_train_base,
    y_test=y_test_base,
    experiment_name="baseline_full_data_full_features",
    sample_fraction=1.0,
    n_features=X_train_base.shape[1],
    extra_info={"strategy": "baseline"},
)

green_results.append(baseline_result)

display(pd.DataFrame([baseline_result]))


,experiment,model,sample_fraction,n_features,train_time_sec,accuracy,precision_weighted,recall_weighted,f1_weighted,roc_auc,strategy
0,baseline_full_data_full_features,LogisticRegression,1.0,63,0.101863,0.915493,0.915493,0.915493,0.915493,0.947619,baseline


## Experimento 1 - Variación del tamaño de muestra

Se analiza cómo cambia el tiempo de entrenamiento y el rendimiento cuando se modifica la fracción de datos utilizada.


In [11]:
# ============================================================
# Experimento 1 - Variación del tamaño de muestra
# ============================================================

sample_fractions = [0.4, 0.6, 0.8, 1.0]

for frac in sample_fractions:
    if frac < 1.0:
        X_sub, _, y_sub, _ = train_test_split(
            X_green,
            y,
            train_size=frac,
            stratify=y,
            random_state=RANDOM_STATE,
        )
    else:
        X_sub, y_sub = X_green.copy(), pd.Series(y).copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X_sub,
        y_sub,
        test_size=0.2,
        stratify=y_sub,
        random_state=RANDOM_STATE,
    )

    result = train_and_evaluate_model(
        model=base_model,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        experiment_name="sample_size_variation",
        sample_fraction=frac,
        n_features=X_train.shape[1],
        extra_info={"strategy": "vary_sample_size"},
    )

    green_results.append(result)

df_sample_results = pd.DataFrame(
    [r for r in green_results if r["experiment"] == "sample_size_variation"]
).sort_values("sample_fraction")

display(df_sample_results)


,experiment,model,sample_fraction,n_features,train_time_sec,accuracy,precision_weighted,recall_weighted,f1_weighted,roc_auc,strategy
0,sample_size_variation,LogisticRegression,0.4,63,0.040133,0.892857,0.894872,0.892857,0.892720,0.897959,vary_sample_size
1,sample_size_variation,LogisticRegression,0.6,63,0.039134,0.761905,0.764302,0.761905,0.761364,0.882086,vary_sample_size
2,sample_size_variation,LogisticRegression,0.8,63,0.049392,0.892857,0.892857,0.892857,0.892857,0.955357,vary_sample_size
3,sample_size_variation,LogisticRegression,1.0,63,0.157568,0.915493,0.915493,0.915493,0.915493,0.947619,vary_sample_size


## Experimento 2 - Variación del número de variables

Se aplica selección de variables para estudiar el efecto de la dimensionalidad sobre el coste computacional y el rendimiento.


In [12]:
# ============================================================
# Experimento 2 - Variación del número de variables
# ============================================================

total_features = X_green.shape[1]

candidate_feature_sizes = sorted(
    list(
        {
            min(3, total_features),
            min(5, total_features),
            min(7, total_features),
            max(1, int(total_features * 0.5)),
            total_features,
        }
    )
)
candidate_feature_sizes = [k for k in candidate_feature_sizes if 1 <= k <= total_features]

feature_selection_results = []

for k in candidate_feature_sizes:
    selector = SelectKBest(score_func=f_classif, k=k)
    X_selected = selector.fit_transform(X_green, y)

    X_train, X_test, y_train, y_test = train_test_split(
        X_selected,
        y,
        test_size=0.2,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    result = train_and_evaluate_model(
        model=base_model,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        experiment_name="feature_count_variation",
        sample_fraction=1.0,
        n_features=k,
        extra_info={"strategy": "select_k_best"},
    )

    green_results.append(result)
    feature_selection_results.append(result)

df_feature_results = pd.DataFrame(feature_selection_results).sort_values("n_features")

display(df_feature_results)


,experiment,model,sample_fraction,n_features,train_time_sec,accuracy,precision_weighted,recall_weighted,f1_weighted,roc_auc,strategy
0,feature_count_variation,LogisticRegression,1.0,3,0.046451,0.563380,0.765398,0.563380,0.455557,0.557143,select_k_best
1,feature_count_variation,LogisticRegression,1.0,5,0.003858,0.591549,0.773781,0.591549,0.505736,0.585714,select_k_best
2,feature_count_variation,LogisticRegression,1.0,7,0.003666,0.633803,0.787369,0.633803,0.574185,0.628571,select_k_best
3,feature_count_variation,LogisticRegression,1.0,31,0.200377,0.915493,0.915493,0.915493,0.915493,0.944444,select_k_best
4,feature_count_variation,LogisticRegression,1.0,63,0.036046,0.915493,0.915493,0.915493,0.915493,0.947619,select_k_best


## Experimento 3 - Regularización

Se evalúa el efecto del parámetro `C` en Logistic Regression como mecanismo de control de complejidad.


In [13]:
# ============================================================
# Experimento 3 - Regularización
# ============================================================

C_values = [0.01, 0.1, 1.0, 10.0]
regularization_results = []

for c_value in C_values:
    regularized_model = LogisticRegression(
        C=c_value,
        max_iter=1000,
        random_state=RANDOM_STATE,
    )

    result = train_and_evaluate_model(
        model=regularized_model,
        X_train=X_train_base,
        X_test=X_test_base,
        y_train=y_train_base,
        y_test=y_test_base,
        experiment_name="regularization_variation",
        sample_fraction=1.0,
        n_features=X_train_base.shape[1],
        extra_info={
            "strategy": "regularization",
            "C_value": c_value,
        },
    )

    green_results.append(result)
    regularization_results.append(result)

df_regularization_results = pd.DataFrame(regularization_results).sort_values("C_value")

display(df_regularization_results)


,experiment,model,sample_fraction,n_features,train_time_sec,accuracy,precision_weighted,recall_weighted,f1_weighted,roc_auc,strategy,C_value
0,regularization_variation,LogisticRegression,1.0,63,0.030343,0.915493,0.915493,0.915493,0.915493,0.939683,regularization,0.01
1,regularization_variation,LogisticRegression,1.0,63,0.017786,0.915493,0.915493,0.915493,0.915493,0.939683,regularization,0.10
2,regularization_variation,LogisticRegression,1.0,63,0.113898,0.915493,0.915493,0.915493,0.915493,0.947619,regularization,1.00
3,regularization_variation,LogisticRegression,1.0,63,0.248342,0.915493,0.915493,0.915493,0.915493,0.965873,regularization,10.00


## Experimento 4 - Sustitución por un modelo más simple

Se compara el baseline con un modelo de árbol poco profundo para evaluar si una estructura más simple puede ofrecer una relación favorable entre rendimiento y coste.


In [14]:
# ============================================================
# Experimento 4 - Modelo alternativo más simple
# ============================================================

simple_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=RANDOM_STATE,
)

simple_model_result = train_and_evaluate_model(
    model=simple_model,
    X_train=X_train_base,
    X_test=X_test_base,
    y_train=y_train_base,
    y_test=y_test_base,
    experiment_name="simple_model_comparison",
    sample_fraction=1.0,
    n_features=X_train_base.shape[1],
    extra_info={"strategy": "simpler_model_max_depth_3"},
)

green_results.append(simple_model_result)

display(pd.DataFrame([simple_model_result]))


,experiment,model,sample_fraction,n_features,train_time_sec,accuracy,precision_weighted,recall_weighted,f1_weighted,roc_auc,strategy
0,simple_model_comparison,DecisionTreeClassifier,1.0,63,0.015345,0.619718,0.785325,0.619718,0.55856,0.625,simpler_model_max_depth_3


## Experimento 5 - Poda en modelos basados en árboles

Se modifica la profundidad máxima del árbol para analizar el efecto de la complejidad estructural sobre el coste de entrenamiento y el desempeño.


In [15]:
# ============================================================
# Experimento 5 - Poda / complejidad del árbol
# ============================================================

tree_depths = [2, 3, 5, None]
tree_pruning_results = []

for depth in tree_depths:
    tree_model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=RANDOM_STATE,
    )

    result = train_and_evaluate_model(
        model=tree_model,
        X_train=X_train_base,
        X_test=X_test_base,
        y_train=y_train_base,
        y_test=y_test_base,
        experiment_name="tree_pruning_variation",
        sample_fraction=1.0,
        n_features=X_train_base.shape[1],
        extra_info={
            "strategy": "tree_pruning",
            "max_depth": depth if depth is not None else "None",
        },
    )

    green_results.append(result)
    tree_pruning_results.append(result)

df_tree_results = pd.DataFrame(tree_pruning_results)

display(df_tree_results)


,experiment,model,sample_fraction,n_features,train_time_sec,accuracy,precision_weighted,recall_weighted,f1_weighted,roc_auc,strategy,max_depth
0,tree_pruning_variation,DecisionTreeClassifier,1.0,63,0.009018,0.619718,0.785325,0.619718,0.558560,0.625000,tree_pruning,2
1,tree_pruning_variation,DecisionTreeClassifier,1.0,63,0.006034,0.619718,0.785325,0.619718,0.558560,0.625000,tree_pruning,3
2,tree_pruning_variation,DecisionTreeClassifier,1.0,63,0.021509,0.647887,0.794601,0.647887,0.600571,0.652778,tree_pruning,5
3,tree_pruning_variation,DecisionTreeClassifier,1.0,63,0.028438,0.873239,0.873586,0.873239,0.873239,0.873413,tree_pruning,None


## Consolidación general de resultados


### Normalización de unidades territoriales

Se identificó que la variable **provincia** presentaba inconsistencias en su representación textual, incluyendo diferencias en mayúsculas, uso de tildes y formatos de escritura. Estas variaciones provocaban una sobreestimación del número de unidades territoriales, elevando artificialmente el conteo de valores únicos hasta 63.

Para corregir este problema, se aplicó un proceso de normalización textual orientado a homogenizar la representación de las provincias, eliminando diferencias de capitalización, tildes y espacios redundantes. Como resultado, fue posible recuperar correctamente las **32 unidades territoriales** del país, correspondientes a las **31 provincias y el Distrito Nacional**.

Este paso es metodológicamente necesario, ya que garantiza la consistencia del análisis territorial y evita interpretaciones erróneas derivadas de duplicados semánticos en los datos.


In [33]:

# ============================================================
# LIMPIEZA Y NORMALIZACION DE LA VARIABLE PROVINCIA
# ============================================================

import unicodedata
import pandas as pd

def normalize_province(text):
    """
    Normaliza nombres de provincias:
    - elimina diferencias de mayúsculas/minúsculas
    - elimina tildes
    - remueve espacios redundantes
    """
    if pd.isna(text):
        return text

    text = str(text).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    text = " ".join(text.split())

    return text

# Se conserva la columna original y se crea una versión normalizada.
df_clean["provincia_clean"] = df_clean["provincia"].apply(normalize_province)

print("Conteo original de valores únicos en 'provincia':", df_clean["provincia"].nunique())
print("Conteo normalizado de valores únicos en 'provincia_clean':", df_clean["provincia_clean"].nunique())

print("\nMuestra de provincias normalizadas:")
print(sorted(df_clean["provincia_clean"].dropna().unique())[:10])


Conteo original de valores únicos en 'provincia': 63
Conteo normalizado de valores únicos en 'provincia_clean': 32

Muestra de provincias normalizadas:
['azua', 'bahoruco', 'barahona', 'dajabon', 'distrito nacional', 'duarte', 'el seibo', 'elias pina', 'espaillat', 'hato mayor']


In [34]:

# ============================================================
# VALIDACION DE UNIDADES TERRITORIALES
# ============================================================

n_unidades = df_clean["provincia_clean"].dropna().nunique()

print(f"Unidades territoriales únicas detectadas tras la normalización: {n_unidades}")

assert n_unidades == 32, (
    "Error: el dataset debe contener exactamente 32 unidades territoriales "
    "(31 provincias y el Distrito Nacional)."
)

print("Validación superada: el dataset contiene 32 unidades territoriales correctas.")


Unidades territoriales únicas detectadas tras la normalización: 32
Validación superada: el dataset contiene 32 unidades territoriales correctas.


In [35]:
# ============================================================
# Consolidación general de resultados
# ============================================================

df_green_results = pd.DataFrame(green_results)

preferred_columns = [
    "experiment",
    "strategy",
    "model",
    "sample_fraction",
    "n_features",
    "train_time_sec",
    "accuracy",
    "precision_weighted",
    "recall_weighted",
    "f1_weighted",
    "roc_auc",
    "C_value",
    "max_depth",
]
existing_columns = [col for col in preferred_columns if col in df_green_results.columns]
remaining_columns = [col for col in df_green_results.columns if col not in existing_columns]
df_green_results = df_green_results[existing_columns + remaining_columns]

print("Resultados consolidados:")
display(df_green_results)


Resultados consolidados:


,experiment,strategy,model,sample_fraction,n_features,train_time_sec,accuracy,precision_weighted,recall_weighted,f1_weighted,roc_auc,C_value,max_depth
0,baseline_full_data_full_features,baseline,LogisticRegression,1.0,63,0.101863,0.915493,0.915493,0.915493,0.915493,0.947619,NaN,NaN
1,sample_size_variation,vary_sample_size,LogisticRegression,0.4,63,0.040133,0.892857,0.894872,0.892857,0.892720,0.897959,NaN,NaN
2,sample_size_variation,vary_sample_size,LogisticRegression,0.6,63,0.039134,0.761905,0.764302,0.761905,0.761364,0.882086,NaN,NaN
3,sample_size_variation,vary_sample_size,LogisticRegression,0.8,63,0.049392,0.892857,0.892857,0.892857,0.892857,0.955357,NaN,NaN
4,sample_size_variation,vary_sample_size,LogisticRegression,1.0,63,0.157568,0.915493,0.915493,0.915493,0.915493,0.947619,NaN,NaN
5,feature_count_variation,select_k_best,LogisticRegression,1.0,3,0.046451,0.563380,0.765398,0.563380,0.455557,0.557143,NaN,NaN
6,feature_count_variation,select_k_best,LogisticRegression,1.0,5,0.003858,0.591549,0.773781,0.591549,0.505736,0.585714,NaN,NaN
7,feature_count_variation,select_k_best,LogisticRegression,1.0,7,0.003666,0.633803,0.787369,0.633803,0.574185,0.628571,NaN,NaN
8,feature_count_variation,select_k_best,LogisticRegression,1.0,31,0.200377,0.915493,0.915493,0.915493,0.915493,0.944444,NaN,NaN
9,feature_count_variation,select_k_best,LogisticRegression,1.0,63,0.036046,0.915493,0.915493,0.915493,0.915493,0.947619,NaN,NaN


In [36]:
# ============================================================
# Resumen ordenado por tiempo de entrenamiento
# ============================================================

df_green_summary = df_green_results.sort_values(
    by=["train_time_sec", "f1_weighted"],
    ascending=[True, False],
).reset_index(drop=True)

display(df_green_summary)


,experiment,strategy,model,sample_fraction,n_features,train_time_sec,accuracy,precision_weighted,recall_weighted,f1_weighted,roc_auc,C_value,max_depth
0,feature_count_variation,select_k_best,LogisticRegression,1.0,7,0.003666,0.633803,0.787369,0.633803,0.574185,0.628571,NaN,NaN
1,feature_count_variation,select_k_best,LogisticRegression,1.0,5,0.003858,0.591549,0.773781,0.591549,0.505736,0.585714,NaN,NaN
2,tree_pruning_variation,tree_pruning,DecisionTreeClassifier,1.0,63,0.006034,0.619718,0.785325,0.619718,0.558560,0.625000,NaN,3
3,tree_pruning_variation,tree_pruning,DecisionTreeClassifier,1.0,63,0.009018,0.619718,0.785325,0.619718,0.558560,0.625000,NaN,2
4,simple_model_comparison,simpler_model_max_depth_3,DecisionTreeClassifier,1.0,63,0.015345,0.619718,0.785325,0.619718,0.558560,0.625000,NaN,NaN
5,regularization_variation,regularization,LogisticRegression,1.0,63,0.017786,0.915493,0.915493,0.915493,0.915493,0.939683,0.10,NaN
6,tree_pruning_variation,tree_pruning,DecisionTreeClassifier,1.0,63,0.021509,0.647887,0.794601,0.647887,0.600571,0.652778,NaN,5
7,tree_pruning_variation,tree_pruning,DecisionTreeClassifier,1.0,63,0.028438,0.873239,0.873586,0.873239,0.873239,0.873413,NaN,None
8,regularization_variation,regularization,LogisticRegression,1.0,63,0.030343,0.915493,0.915493,0.915493,0.915493,0.939683,0.01,NaN
9,feature_count_variation,select_k_best,LogisticRegression,1.0,63,0.036046,0.915493,0.915493,0.915493,0.915493,0.947619,NaN,NaN


In [37]:
# ============================================================
# Visualizaciones
# ============================================================

# Esta sección centraliza la generación y exportación de figuras
# para evitar inconsistencias entre celdas, variables no definidas
# o capturas manuales.
# Todas las figuras se guardan automáticamente en formato PNG
# dentro de 'reports/figures'

In [38]:
# ============================================================
# Utilidades centralizadas para generación y guardado de figuras
# ============================================================

from pathlib import Path
from IPython.display import Image, Markdown, display
import json

FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def save_line_plot(
    df,
    x_col,
    y_col,
    filename,
    title,
    xlabel,
    ylabel,
    figsize=(8, 5),
):
    """Genera un gráfico de línea, lo guarda en PNG y lo muestra."""
    if df.empty:
        raise ValueError(f"No se puede generar '{filename}' porque el DataFrame está vacío.")

    plot_df = (
        df[[x_col, y_col]]
        .dropna()
        .drop_duplicates()
        .sort_values(x_col)
    )

    if plot_df.empty:
        raise ValueError(f"No hay datos válidos para generar '{filename}'.")

    output_path = FIGURES_DIR / filename

    plt.figure(figsize=figsize)
    plt.plot(
        plot_df[x_col],
        plot_df[y_col],
        marker="o",
    )
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    return output_path


def save_scatter_plot(
    df,
    x_col,
    y_col,
    filename,
    title,
    xlabel,
    ylabel,
    label_col=None,
    figsize=(10, 6),
):
    """Genera un scatter plot, lo guarda en PNG y lo muestra."""
    if df.empty:
        raise ValueError(f"No se puede generar '{filename}' porque el DataFrame está vacío.")

    cols = [x_col, y_col]
    if label_col:
        cols.append(label_col)

    plot_df = df[cols].dropna(subset=[x_col, y_col]).copy()

    if plot_df.empty:
        raise ValueError(f"No hay datos válidos para generar '{filename}'.")

    output_path = FIGURES_DIR / filename

    plt.figure(figsize=figsize)
    plt.scatter(
        plot_df[x_col],
        plot_df[y_col],
    )

    if label_col:
        for _, row in plot_df.iterrows():
            plt.annotate(
                str(row[label_col]),
                (row[x_col], row[y_col]),
                fontsize=8,
                alpha=0.8,
            )

    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    return output_path


def export_figure(fig, stem: str, dpi: int = 180):
    """Exporta una figura en PNG, PDF, SVG y JSON de metadatos."""
    png_path = FIGURES_DIR / f"{stem}.png"
    pdf_path = FIGURES_DIR / f"{stem}.pdf"
    svg_path = FIGURES_DIR / f"{stem}.svg"
    meta_path = FIGURES_DIR / f"{stem}_metadata.json"

    fig.savefig(png_path, dpi=dpi, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    fig.savefig(svg_path, bbox_inches="tight", facecolor="white")

    metadata = {
        "stem": stem,
        "png": str(png_path),
        "pdf": str(pdf_path),
        "svg": str(svg_path),
    }
    meta_path.write_text(
        json.dumps(metadata, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    return {
        "png": str(png_path),
        "pdf": str(pdf_path),
        "svg": str(svg_path),
        "metadata": str(meta_path),
    }


def display_single_figure_png(export_result: dict, title: str, width: int = 900):
    """Muestra únicamente el PNG exportado para evitar renders rotos con HTML."""
    png_path = Path(export_result["png"])

    if not png_path.exists():
        raise FileNotFoundError(f"No se encontró la imagen: {png_path}")

    display(Markdown(f"### {title}"))
    display(Image(filename=str(png_path), width=width))

    print("Archivos exportados:")
    print(f"png: {export_result['png']}")
    print(f"pdf: {export_result['pdf']}")
    print(f"svg: {export_result['svg']}")
    print(f"metadata: {export_result['metadata']}")

print("Carpeta de figuras:", FIGURES_DIR)


Carpeta de figuras: /content/transport-ml-rd/reports/figures


In [39]:
# ============================================================
# Figura 1 - Trade-off entre F1-score ponderado y tiempo de entrenamiento
# ============================================================

df_tradeoff = df_green_results.copy()
df_tradeoff["plot_label"] = df_tradeoff["model"] + " | " + df_tradeoff["strategy"]

plot_df = (
    df_tradeoff[["train_time_sec", "f1_weighted", "plot_label", "model", "strategy"]]
    .dropna(subset=["train_time_sec", "f1_weighted"])
    .copy()
)

if plot_df.empty:
    raise ValueError("No hay datos válidos para construir la figura de trade-off.")

# Selección de pocos puntos a anotar para evitar solapamiento visual
best_global_idx = plot_df["f1_weighted"].idxmax()
fastest_idx = plot_df["train_time_sec"].idxmin()

high_perf_df = (
    plot_df.sort_values(
        by=["f1_weighted", "train_time_sec"],
        ascending=[False, True]
    )
    .head(5)
    .copy()
)

label_candidates = plot_df.loc[[best_global_idx, fastest_idx]].copy()
label_candidates = (
    pd.concat([label_candidates, high_perf_df])
    .drop_duplicates(subset=["plot_label"])
    .sort_values(by=["f1_weighted", "train_time_sec"], ascending=[False, True])
    .head(3)
    .copy()
)

fig, ax = plt.subplots(figsize=(11, 6.5))

ax.scatter(
    plot_df["train_time_sec"],
    plot_df["f1_weighted"],
    s=70,
)

for i, (_, row) in enumerate(label_candidates.iterrows(), start=1):
    ax.annotate(
        f"[{i}]",
        (row["train_time_sec"], row["f1_weighted"]),
        fontsize=9,
        xytext=(6, 6),
        textcoords="offset points",
    )

ax.set_title("Trade-off entre F1-score ponderado y tiempo de entrenamiento", fontsize=14)
ax.set_xlabel("Tiempo de entrenamiento (segundos)", fontsize=12)
ax.set_ylabel("F1-score ponderado", fontsize=12)
ax.grid(True, alpha=0.3)

x_min = max(0, plot_df["train_time_sec"].min() - 0.01)
x_max = plot_df["train_time_sec"].max() + 0.03
y_min = max(0, plot_df["f1_weighted"].min() - 0.02)
y_max = min(1.0, plot_df["f1_weighted"].max() + 0.02)

ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)

fig.tight_layout()

export_result = export_figure(fig, "figure_1_tradeoff", dpi=180)
display_single_figure_png(
    export_result,
    title="Figura 1. Trade-off entre F1-score ponderado y tiempo de entrenamiento",
    width=1000,
)

plt.close(fig)

print("Referencias de anotaciones mostradas en la figura:")
for i, (_, row) in enumerate(label_candidates.iterrows(), start=1):
    print(
        f"[{i}] Modelo={row['model']} | estrategia={row['strategy']} | "
        f"train_time_sec={row['train_time_sec']:.4f} | f1_weighted={row['f1_weighted']:.4f}"
    )

print("\nResumen de configuraciones con mejor desempeño:")
display(
    df_tradeoff.sort_values(
        by=["f1_weighted", "train_time_sec"],
        ascending=[False, True]
    )[["model", "strategy", "train_time_sec", "f1_weighted"]].head(10)
)


Figura 1 guardada en: /content/transport-ml-rd/reports/figures/figura_1_tradeoff_f1_vs_tiempo.png
Puntos destacados para análisis:
             model       strategy  train_time_sec  f1_weighted
LogisticRegression regularization        0.017786     0.915493
LogisticRegression regularization        0.030343     0.915493
LogisticRegression  select_k_best        0.036046     0.915493
LogisticRegression       baseline        0.101863     0.915493
LogisticRegression regularization        0.113898     0.915493


In [40]:
# ============================================================
# Figura 2 - F1-score ponderado vs fracción de la muestra
# ============================================================

fig2_path = save_line_plot(
    df=df_sample_results,
    x_col="sample_fraction",
    y_col="f1_weighted",
    filename="figura_2_f1_vs_fraccion_muestra.png",
    title="F1-score ponderado vs. fracción de la muestra",
    xlabel="Fracción de muestra",
    ylabel="F1-score ponderado",
)

print("Figura 2 guardada en:", fig2_path)


Figura 2 guardada en: /content/transport-ml-rd/reports/figures/figura_2_f1_vs_fraccion_muestra.png


In [41]:
# ============================================================
# Figura 3 - Tiempo de entrenamiento vs número de variables
# ============================================================

fig3_path = save_line_plot(
    df=df_feature_results,
    x_col="n_features",
    y_col="train_time_sec",
    filename="figura_3_tiempo_vs_numero_variables.png",
    title="Tiempo de entrenamiento vs. número de variables",
    xlabel="Número de variables",
    ylabel="Tiempo de entrenamiento (segundos)",
)

print("Figura 3 guardada en:", fig3_path)


Figura 3 guardada en: /content/transport-ml-rd/reports/figures/figura_3_tiempo_vs_numero_variables.png


In [42]:
# ============================================================
# Figura 4 - F1-score ponderado vs número de variables
# ============================================================

fig4_path = save_line_plot(
    df=df_feature_results,
    x_col="n_features",
    y_col="f1_weighted",
    filename="figura_4_f1_vs_numero_variables.png",
    title="F1-score ponderado vs. número de variables",
    xlabel="Número de variables",
    ylabel="F1-score ponderado",
)

print("Figura 4 guardada en:", fig4_path)

# ============================================================
# Figura opcional - Tiempo de entrenamiento vs fracción de la muestra
# ============================================================

fig_extra_path = save_line_plot(
    df=df_sample_results,
    x_col="sample_fraction",
    y_col="train_time_sec",
    filename="figura_extra_tiempo_vs_fraccion_muestra.png",
    title="Tiempo de entrenamiento vs. fracción de la muestra",
    xlabel="Fracción de muestra",
    ylabel="Tiempo de entrenamiento (segundos)",
)

print("Figura extra guardada en:", fig_extra_path)

# ============================================================
# Resumen de figuras exportadas
# ============================================================

figures_summary = pd.DataFrame(
    [
        {"Figura": "Figura 1", "Archivo": "figura_1_tradeoff_f1_vs_tiempo.png", "Descripción": "Trade-off F1 vs tiempo"},
        {"Figura": "Figura 2", "Archivo": "figura_2_f1_vs_fraccion_muestra.png", "Descripción": "F1 vs fracción de muestra"},
        {"Figura": "Figura 3", "Archivo": "figura_3_tiempo_vs_numero_variables.png", "Descripción": "Tiempo vs número de variables"},
        {"Figura": "Figura 4", "Archivo": "figura_4_f1_vs_numero_variables.png", "Descripción": "F1 vs número de variables"},
        {"Figura": "Figura extra", "Archivo": "figura_extra_tiempo_vs_fraccion_muestra.png", "Descripción": "Tiempo vs fracción de muestra"},
    ]
)

display(figures_summary)
print("Carpeta de figuras:", FIGURES_DIR)


Figura 4 guardada en: /content/transport-ml-rd/reports/figures/figura_4_f1_vs_numero_variables.png
Figura extra guardada en: /content/transport-ml-rd/reports/figures/figura_extra_tiempo_vs_fraccion_muestra.png


,Figura,Archivo,Descripción
0,Figura 1,figura_1_tradeoff_f1_vs_tiempo.png,Trade-off F1 vs tiempo
1,Figura 2,figura_2_f1_vs_fraccion_muestra.png,F1 vs fracción de muestra
2,Figura 3,figura_3_tiempo_vs_numero_variables.png,Tiempo vs número de variables
3,Figura 4,figura_4_f1_vs_numero_variables.png,F1 vs número de variables
4,Figura extra,figura_extra_tiempo_vs_fraccion_muestra.png,Tiempo vs fracción de muestra


Carpeta de figuras: /content/transport-ml-rd/reports/figures


## Selección de configuración Green AI recomendada

Criterio de trabajo: entre las configuraciones cuyo F1 sea al menos el 95% del mejor F1 observado, seleccionar la de menor tiempo de entrenamiento.


In [26]:
# ============================================================
# Selección de mejor configuración Green AI
# ============================================================

best_f1 = df_green_results["f1_weighted"].max()
threshold = best_f1 * 0.95

df_candidate_green = df_green_results[
    df_green_results["f1_weighted"] >= threshold
].copy()

df_candidate_green = df_candidate_green.sort_values("train_time_sec").reset_index(drop=True)

print("Mejor F1 global:", best_f1)
print("Umbral Green AI (95% del mejor F1):", threshold)

display(df_candidate_green)

if not df_candidate_green.empty:
    recommended_green_config = df_candidate_green.iloc[0]
    print("Configuración Green AI recomendada:")
    display(pd.DataFrame([recommended_green_config]))


Mejor F1 global: 0.9154929577464789
Umbral Green AI (95% del mejor F1): 0.8697183098591549


,experiment,strategy,model,sample_fraction,n_features,train_time_sec,accuracy,precision_weighted,recall_weighted,f1_weighted,roc_auc,C_value,max_depth
0,regularization_variation,regularization,LogisticRegression,1.0,63,0.017786,0.915493,0.915493,0.915493,0.915493,0.939683,0.10,NaN
1,tree_pruning_variation,tree_pruning,DecisionTreeClassifier,1.0,63,0.028438,0.873239,0.873586,0.873239,0.873239,0.873413,NaN,None
2,regularization_variation,regularization,LogisticRegression,1.0,63,0.030343,0.915493,0.915493,0.915493,0.915493,0.939683,0.01,NaN
3,feature_count_variation,select_k_best,LogisticRegression,1.0,63,0.036046,0.915493,0.915493,0.915493,0.915493,0.947619,NaN,NaN
4,sample_size_variation,vary_sample_size,LogisticRegression,0.4,63,0.040133,0.892857,0.894872,0.892857,0.892720,0.897959,NaN,NaN
5,sample_size_variation,vary_sample_size,LogisticRegression,0.8,63,0.049392,0.892857,0.892857,0.892857,0.892857,0.955357,NaN,NaN
6,baseline_full_data_full_features,baseline,LogisticRegression,1.0,63,0.101863,0.915493,0.915493,0.915493,0.915493,0.947619,NaN,NaN
7,regularization_variation,regularization,LogisticRegression,1.0,63,0.113898,0.915493,0.915493,0.915493,0.915493,0.947619,1.00,NaN
8,sample_size_variation,vary_sample_size,LogisticRegression,1.0,63,0.157568,0.915493,0.915493,0.915493,0.915493,0.947619,NaN,NaN
9,feature_count_variation,select_k_best,LogisticRegression,1.0,31,0.200377,0.915493,0.915493,0.915493,0.915493,0.944444,NaN,NaN


Configuración Green AI recomendada:


,experiment,strategy,model,sample_fraction,n_features,train_time_sec,accuracy,precision_weighted,recall_weighted,f1_weighted,roc_auc,C_value,max_depth
0,regularization_variation,regularization,LogisticRegression,1.0,63,0.017786,0.915493,0.915493,0.915493,0.915493,0.939683,0.1,NaN


In [27]:
# ============================================================
# Comparación final entre baseline, mejor Green AI y modelo simple
# ============================================================

comparison_rows = [baseline_result]

if "recommended_green_config" in globals():
    comparison_rows.append(recommended_green_config.to_dict())

comparison_rows.append(simple_model_result)

df_final_comparison = pd.DataFrame(comparison_rows).drop_duplicates()

display(df_final_comparison)


,experiment,model,sample_fraction,n_features,train_time_sec,accuracy,precision_weighted,recall_weighted,f1_weighted,roc_auc,strategy,C_value,max_depth
0,baseline_full_data_full_features,LogisticRegression,1.0,63,0.101863,0.915493,0.915493,0.915493,0.915493,0.947619,baseline,NaN,NaN
1,regularization_variation,LogisticRegression,1.0,63,0.017786,0.915493,0.915493,0.915493,0.915493,0.939683,regularization,0.1,NaN
2,simple_model_comparison,DecisionTreeClassifier,1.0,63,0.015345,0.619718,0.785325,0.619718,0.558560,0.625000,simpler_model_max_depth_3,NaN,NaN


In [28]:
# ============================================================
# Guardado de resultados
# ============================================================

reports_dir = PROJECT_ROOT / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

green_results_path = reports_dir / "green_ai_results.csv"
green_summary_path = reports_dir / "green_ai_summary.csv"
green_final_comparison_path = reports_dir / "green_ai_final_comparison.csv"

df_green_results.to_csv(green_results_path, index=False)
df_green_summary.to_csv(green_summary_path, index=False)
df_final_comparison.to_csv(green_final_comparison_path, index=False)

print("Archivos guardados:")
print(green_results_path)
print(green_summary_path)
print(green_final_comparison_path)


Archivos guardados:
/content/transport-ml-rd/reports/green_ai_results.csv
/content/transport-ml-rd/reports/green_ai_summary.csv
/content/transport-ml-rd/reports/green_ai_final_comparison.csv


In [29]:
# ============================================================
# Síntesis descriptiva inicial
# ============================================================

print("Resumen descriptivo inicial")
print("-" * 40)
print(f"Total de experimentos ejecutados: {len(df_green_results)}")
print(f"Mejor F1 ponderado observado: {df_green_results['f1_weighted'].max():.4f}")
print(f"Menor tiempo de entrenamiento observado: {df_green_results['train_time_sec'].min():.6f} segundos")
print(f"Mayor tiempo de entrenamiento observado: {df_green_results['train_time_sec'].max():.6f} segundos")

fastest_row = df_green_results.sort_values("train_time_sec").iloc[0]
best_f1_row = df_green_results.sort_values("f1_weighted", ascending=False).iloc[0]

print("\nConfiguración más rápida:")
display(pd.DataFrame([fastest_row]))

print("Configuración con mejor F1:")
display(pd.DataFrame([best_f1_row]))


Resumen descriptivo inicial
----------------------------------------
Total de experimentos ejecutados: 19
Mejor F1 ponderado observado: 0.9155
Menor tiempo de entrenamiento observado: 0.003666 segundos
Mayor tiempo de entrenamiento observado: 0.248342 segundos

Configuración más rápida:


,experiment,strategy,model,sample_fraction,n_features,train_time_sec,accuracy,precision_weighted,recall_weighted,f1_weighted,roc_auc,C_value,max_depth
7,feature_count_variation,select_k_best,LogisticRegression,1.0,7,0.003666,0.633803,0.787369,0.633803,0.574185,0.628571,NaN,NaN


Configuración con mejor F1:


,experiment,strategy,model,sample_fraction,n_features,train_time_sec,accuracy,precision_weighted,recall_weighted,f1_weighted,roc_auc,C_value,max_depth
0,baseline_full_data_full_features,baseline,LogisticRegression,1.0,63,0.101863,0.915493,0.915493,0.915493,0.915493,0.947619,NaN,NaN


### Figura 1. Análisis del trade-off entre desempeño y costo computacional

La Figura 1 presenta el compromiso (trade-off) entre el desempeño del modelo, medido mediante el F1-score ponderado, y el costo computacional asociado, representado por el tiempo de entrenamiento.

Para enriquecer la interpretación, se incorpora la **frontera de Pareto**, la cual identifica aquellas configuraciones que no son dominadas por otras, es decir, aquellas que ofrecen el mejor equilibrio posible entre ambas dimensiones sin que exista otra alternativa que mejore simultáneamente el desempeño y reduzca el tiempo.

Adicionalmente, se realiza la **detección automática del mejor modelo global**, bajo un criterio de optimización que prioriza la maximización del F1-score y, en caso de empate, la minimización del tiempo de entrenamiento. Este modelo es resaltado explícitamente en la figura para facilitar su identificación dentro del conjunto de soluciones evaluadas.

Este enfoque permite una evaluación más rigurosa y estructurada del comportamiento de los modelos, alineada con prácticas de análisis comparativo en contextos de machine learning aplicado.


In [30]:

# ============================================================
# VALIDACIÓN DE UNIDADES TERRITORIALES
# ============================================================

# Ajusta el nombre de la columna si es diferente (ej. 'provincia')
col_territorial = None
for c in df_green_results.columns:
    if c.lower() in ["provincia", "province", "region"]:
        col_territorial = c
        break

if col_territorial is None:
    print("No se encontró columna territorial automáticamente. Ajusta 'col_territorial'.")
else:
    n_unidades = df_green_results[col_territorial].nunique()
    print(f"Unidades territoriales únicas detectadas: {n_unidades}")
    print("Nota: En RD deben ser 32 (31 provincias + Distrito Nacional).")


No se encontró columna territorial automáticamente. Ajusta 'col_territorial'.
